In [ ]:

import SimpleITK as sitk  
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
import ipywidgets as widgets
import pandas as pd
from monai.transforms import RemoveSmallObjects, KeepLargestConnectedComponent
import blosc2
# ============================================================
# Utility Functions
# ============================================================

def load_b2nd(path):
    """Load a .b2nd file as a numpy array using blosc2."""
    schunk = blosc2.open(path, mode="r")
    arr = schunk[:][0]  # Load full array
    return arr

def load_volume(path):
    """Load a 3D volume from NIfTI (.nii.gz), NPZ, NPY, or B2ND files."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    if path.suffix == ".npy":
        return np.load(path)
    elif path.suffix == ".npz":
        data = np.load(path)
        key = list(data.keys())[0]
        return data[key]
    elif path.suffix in [".nii", ".gz"]:
        return nib.load(str(path)).get_fdata()
    elif path.suffix == ".b2nd":
        return load_b2nd(path)
    else:
        raise ValueError(f"Unsupported format: {path.suffix}")

def resample_label_to_image(label_path, image_ref_path):
    """Resample label to match reference image (CT or input image)."""
    label_sitk = sitk.ReadImage(str(label_path))
    ref_sitk = sitk.ReadImage(str(image_ref_path))
    label_resampled = sitk.Resample(
        label_sitk,
        ref_sitk,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        label_sitk.GetPixelID()
    )
    print(f"Resampled label: {label_resampled.GetSize()} to match CT: {ref_sitk.GetSize()}")
    return sitk.GetArrayFromImage(label_resampled).transpose(2, 1, 0)

def window_ct_hu(ct_hu, level=50, width=350):
    lower, upper = level - width / 2.0, level + width / 2.0
    ct_clipped = np.clip(ct_hu, lower, upper)
    return (ct_clipped - lower) / (upper - lower + 1e-6)

def get_slice(volume, axis, idx):
    if axis == "axial":
        return volume[:, :, idx]
    elif axis == "coronal":
        return volume[:, idx, :]
    elif axis == "sagittal":
        return volume[idx, :, :]
    else:
        raise ValueError(f"Invalid axis: {axis}. Must be 'axial', 'coronal', or 'sagittal'.")

# ============================================================
# Overlay Builders
# ============================================================

def build_label_overlay(label_slice):
    unique_labels = np.unique(label_slice)
    if np.array_equal(unique_labels, [0]) or np.array_equal(unique_labels, [0, 1]):
        cmap = ListedColormap([[0,0,0,0], [1,0,0,0.45]])
        return label_slice.astype(np.int32), cmap, (0, 1)
    colors = [[0, 0, 0, 0]]
    rng = np.random.default_rng(42)
    for _ in range(int(unique_labels.max())):
        colors.append([*rng.random(3), 0.45])
    cmap = ListedColormap(colors)
    return label_slice.astype(np.int32), cmap, (0, int(unique_labels.max()))

def build_label_overlay_small_objects_3d(label_volume, min_size=200, connectivity=2):
    delete_small = RemoveSmallObjects(min_size=min_size, connectivity=connectivity)
    cleaned_label = delete_small(label_volume)
    small_objects = (label_volume > 0) & (cleaned_label == 0)
    large_objects = (cleaned_label > 0)
    overlay = np.zeros_like(label_volume, dtype=np.int32)
    overlay[large_objects] = 1
    overlay[small_objects] = 2
    cmap = ListedColormap([[0,0,0,0],[0,1,0,0.25],[1,0,0,0.25]])
    return overlay, cmap, (0, 2)

def build_label_overlay_largest_3d(label_volume):
    largest_component = KeepLargestConnectedComponent(connectivity=2)(label_volume)
    overlay = np.zeros_like(label_volume, dtype=np.int32)
    overlay[label_volume > 0] = 1
    overlay[largest_component > 0] = 2
    cmap = ListedColormap([[0,0,0,0],[1,0,0,0.25],[0,1,0,0.25]])
    return overlay, cmap, (0, 2)

# ============================================================
# Visualization
# ============================================================

def visualize_case(ct_path, label_path, uid, target,
                   axis="axial", mode="labels",
                   window_level=50, window_width=350,
                   save_dir=None, save_all_slices=False,
                   resample_labels=False):
    """
    Visualize one case with selectable mode:
      - 'labels', 'large_component', 'small_objects'
    Optionally resample labels to match CT grid.
    """
    ct = load_volume(ct_path)
    if resample_labels:
        label = resample_label_to_image(label_path, ct_path)
    else:
        label = load_volume(label_path)
        
    print(f"Loaded UID {uid}: CT shape {ct.shape}, Label shape {label.shape}")
    assert ct.shape == label.shape, f"Shape mismatch for {uid}"

    annotated_slices = np.where(np.any(label > 0, axis=(0, 1)))[0]
    if len(annotated_slices) > 0:
        print(f"Annotations at slices: {annotated_slices}")
    else:
        print("No annotations found.")

    if mode == "large_component":
        overlay_3d, cmap, vminmax = build_label_overlay_largest_3d(label)
    elif mode == "small_objects":
        overlay_3d, cmap, vminmax = build_label_overlay_small_objects_3d(label)
    else:
        overlay_3d, cmap, vminmax = label, *build_label_overlay(label[:, :, 0])[1:]

    axis_to_dim = {"axial": 2, "coronal": 1, "sagittal": 0}
    n_slices = ct.shape[axis_to_dim[axis]]

    out_uid_dir = None
    if save_all_slices and save_dir:
        out_uid_dir = Path(save_dir) / str(uid)
        out_uid_dir.mkdir(parents=True, exist_ok=True)

    def plot_slice(idx):
        ct_slice = get_slice(ct, axis, idx)
        overlay_slice = get_slice(overlay_3d, axis, idx)
        ct_img = window_ct_hu(ct_slice, window_level, window_width)

        plt.figure(figsize=(6,6))
        plt.imshow(ct_img.T, cmap="gray", origin="lower")
        plt.imshow(overlay_slice.T, cmap=cmap, origin="lower",
                   vmin=vminmax[0], vmax=vminmax[1])
        plt.axis("off")
        plt.title(f"UID {uid} | {target} | {axis.capitalize()} slice {idx}/{n_slices}")
        plt.tight_layout()

        if out_uid_dir:
            plt.savefig(out_uid_dir / f"{axis}_slice{idx:03d}.png", bbox_inches='tight')
            plt.close()
        else:
            plt.show()

    slice_slider = widgets.IntSlider(
        value=n_slices//2, min=0, max=n_slices-1, step=1,
        description=f'UID {uid}', continuous_update=False
    )
    widgets.interact(plot_slice, idx=slice_slider)

    if save_all_slices and out_uid_dir:
        print(f"Saving all slices for UID {uid} → {out_uid_dir}")
        for i in range(n_slices):
            plot_slice(i)

            plot_slice(i)
# ============================
# Main Visualization Function
# ============================


def visualize_dataset(uids, img_root, label_root,resample_labels=False,mode="label", axis="axial", save_dir=None, save_all_slices=False):
    """Visualize multiple UIDs."""
    img_root, label_root = Path(img_root), Path(label_root)
    df = pd.read_csv(f"/data/colon_cancer/Classifier/ColonCancer/splits.csv")
    for uid in uids:
        # try naming patterns
        possible_img_names = [
            f"{uid}.nii.gz",f"{uid:03d}.nii.gz", f"{uid}_0000.nii.gz", f"{uid:03d}_0000.nii.gz",
            f"colon_{uid:03d}.nii.gz", f"{uid}.npy", f"{uid}.npz",  f"{uid:03d}.b2nd",
        ]
        possible_label_names = [
            f"{uid}.nii.gz",f"{uid:03d}.nii.gz", f"{uid}_0000.nii.gz", f"{uid:03d}_0000.nii.gz",
            f"colon_{uid:03d}.nii.gz", f"{uid}.npy", f"{uid}.npz",  f"{uid:03d}_seg.b2nd", 
        ]
        
        
        img_path = next((img_root / n for n in possible_img_names if (img_root / n).exists()), None)
        label_path = next((label_root / n for n in possible_label_names if (label_root / n).exists()), None)
        print(img_path, label_path)
        if img_path is None or label_path is None:
            print(f"Skipping UID {uid}: missing image or label")
            continue

        row = df[df["UID"] == uid]
        target = row["target"].values[0]
        if target == 0:
            target = "div"
        else:
            target = "cc"

        visualize_case(img_path, label_path, uid,mode=mode, axis=axis, target=target,
                       save_dir=save_dir, save_all_slices=save_all_slices,resample_labels=resample_labels)


# ============================
# Example usage
# ============================

if __name__ == "__main__":
    #108, 565, 752, 803, 370
    uids = [370  ]
    
    """
    #img_root = "/data/colon_cancer/Classifier/pp_Tr_npz"
    #label_root = "/data/colon_cancer/Classifier/resampledTr/labels_resampled"
    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/nnUNet_results/Dataset101_CC/nnUNetTrainer__nnUNetResEncUNetLPlans__3d_fullres/fold_0/validation"
    save_dir = "/data/benchaaben/classifier/ct_overlays/finetuned"

    visualize_dataset(uids, img_root, label_root,
                    axis="axial", save_dir=save_dir, mode ="label", save_all_slices=False)
    
    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/final"
    save_dir = "/data/benchaaben/classifier/ct_overlays/pretrained"
    visualize_dataset(uids, img_root, label_root,
                      axis="axial", save_dir=save_dir,mode ="label",resample_labels=True, save_all_slices=False)


    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/original"
    save_dir = "/data/benchaaben/classifier/ct_overlays/pretrained"
    visualize_dataset(uids, img_root, label_root,mode ="label",
                      axis="axial", save_dir=save_dir, save_all_slices=False)
    
    img_root = "/data/colon_cancer/nnUNet_raw/Dataset100_CC/imagesTr"
    label_root = "/data/colon_cancer/nnUNet_raw/Dataset100_CC/labelsTr"
    save_dir = "/data/benchaaben/classifier/ct_overlays/original"
    visualize_dataset(uids, img_root, label_root,mode ="label",
                      axis="axial", save_dir=save_dir, save_all_slices=False)
    """
    ##input to segmentator
    img_root = f"/data/colon_cancer/Classifier/ColonCancer/nnUNetPlans_3d_fullres"
    label_root = f"/data/colon_cancer/Classifier/ColonCancer/nnUNetPlans_3d_fullres"
    visualize_dataset(uids, img_root, label_root,mode ="label",
                      axis="axial", save_all_slices=False)
    
    ##input to classifier 
    img_root = f"/data/colon_cancer/Classifier/ColonCancer/resampledTr/images_resampled"
    label_root = f"/data/colon_cancer/Classifier/ColonCancer/resampledTr/labels_resampled"
    visualize_dataset(uids, img_root, label_root,mode ="label",
                      axis="axial", save_all_slices=False)
    


In [ ]:
import pickle
path=f"/data/colon_cancer/nnUNet_preprocessed/Dataset100_CC/nnUNetPlans_3d_fullres/001.pkl"
with open(path, "rb") as f:
    arr = pickle.load(f)
print(arr)

In [2]:

import SimpleITK as sitk  
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
import ipywidgets as widgets
import pandas as pd
from monai.transforms import RemoveSmallObjects, KeepLargestConnectedComponent
import blosc2



import numpy as np
from pathlib import Path
from typing import Sequence, Tuple, List
from tqdm import tqdm
import json
import numpy as np
import pandas as pd
import nibabel as nib
from pathlib import Path
from typing import Tuple, List
import numpy as np


def voxels_from_mm(spacing, margin_mm: Tuple[float, float, float]):
    """Convert physical margin in mm to voxel units (rounding up)."""
    return tuple([int(np.ceil(mm / sp)) for mm, sp in zip(margin_mm, spacing)])


def get_bbox_from_mask_with_margin(mask: np.ndarray, margin_vox: Tuple[int, int, int]) -> List[Tuple[int, int]]:
    """Compute bounding box from nonzero mask with added voxel margin."""
    nonzero = np.where(mask > 0)
    if len(nonzero[0]) == 0:
        # no label found
        return [(0, mask.shape[0]), (0, mask.shape[1]), (0, mask.shape[2])]
    
    bbox = []
    for d in range(3):
        start = max(int(np.min(nonzero[d])) - margin_vox[d], 0)
        end = min(int(np.max(nonzero[d])) + margin_vox[d] + 1, mask.shape[d])
        bbox.append((start, end))
    return bbox


def crop_to_bbox_no_channels(image, bbox: Sequence[Sequence[int]]):
    """Crops 3D image to bounding box (no channels)."""
    resizer = tuple(slice(start, end) for start, end in bbox)
    return image[resizer]


def crop_to_bbox(data: np.ndarray, bbox: Sequence[Sequence[int]]):
    """Crops 3D/4D array per channel to given bounding box."""
    cropped_data = [crop_to_bbox_no_channels(data[c], bbox) for c in range(data.shape[0])]
    return np.stack(cropped_data)


def crop_to_label_region(data: np.ndarray,
                         seg: np.ndarray,
                         spacing: Tuple[float, float, float],
                         margin_min: float = 15.0) -> Tuple[np.ndarray, np.ndarray, List[Tuple[int, int]]]:
    """
    Crop image & segmentation to a region of interest defined by the label (segmentation mask).

    Args:
        data (np.ndarray): 4D image [C, D, H, W]
        seg (np.ndarray): 3D segmentation [D, H, W]
        spacing (Tuple[float]): voxel spacing in mm
        margin_min (float): margin (in mm) added around the labeled region

    Returns:
        cropped_data (np.ndarray): cropped image
        cropped_seg (np.ndarray): cropped segmentation
        bbox (List[Tuple[int, int]]): voxel bounding box
    """
    margin_vox = voxels_from_mm(spacing, (margin_min, margin_min, margin_min))
    bbox = get_bbox_from_mask_with_margin(seg, margin_vox)

    data_cropped = crop_to_bbox(data, bbox)
    seg_cropped = crop_to_bbox_no_channels(seg, bbox)

    return data_cropped, seg_cropped, bbox

# ============================================================
# Utility Functions
# ============================================================

def load_b2nd(path):
    """Load a .b2nd file as a numpy array using blosc2."""
    schunk = blosc2.open(path, mode="r")
    arr = schunk[:][0]  # Load full array
    return arr

def load_volume(path):
    """Load a 3D volume from NIfTI (.nii.gz), NPZ, NPY, or B2ND files."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    if path.suffix == ".npy":
        return np.load(path)
    elif path.suffix == ".npz":
        data = np.load(path)
        key = list(data.keys())[0]
        return data[key]
    elif path.suffix in [".nii", ".gz"]:
        return nib.load(str(path)).get_fdata()
    elif path.suffix == ".b2nd":
        return load_b2nd(path)
    else:
        raise ValueError(f"Unsupported format: {path.suffix}")
def resample_image_to_spacing(image_path, target_spacing=(1.0, 1.0, 1.0), is_label=False):
    """
    Resample an image (or label) to a target voxel spacing.
    Works with NIfTI, NPZ, NPY, or B2ND files.
    Uses linear interpolation for images, nearest neighbor for labels.
    """
    path = Path(image_path)
    
    # Try to read with SimpleITK (works for .nii, .nii.gz, etc.)
    if path.suffix in [".nii", ".nii.gz", ".mha", ".mhd"]:
        sitk_image = sitk.ReadImage(str(path))
        original_spacing = np.array(sitk_image.GetSpacing())
        original_size = np.array(sitk_image.GetSize())
    else:
        # For .b2nd, .npy, .npz → load via numpy, then wrap into SimpleITK image
        arr = load_volume(path)
        sitk_image = sitk.GetImageFromArray(arr.transpose(2, 1, 0))  # match orientation
        # Assign a fake default spacing if not known
        original_spacing = np.array( [
                0.73828125,
                0.73828125,
                0.8999999761581421,
            ])
        original_size = np.array(sitk_image.GetSize())

    new_spacing = np.array(target_spacing, float)
    new_size = np.round(original_size * (original_spacing / new_spacing)).astype(int)

    interpolator = sitk.sitkNearestNeighbor if is_label else sitk.sitkLinear

    resampled = sitk.Resample(
        sitk_image,
        size=list(map(int, new_size)),
        transform=sitk.Transform(),
        interpolator=interpolator,
        outputOrigin=sitk_image.GetOrigin(),
        outputSpacing=tuple(new_spacing),
        outputDirection=sitk_image.GetDirection(),
        defaultPixelValue=0,
        outputPixelType=sitk_image.GetPixelID()
    )

    print(f"Resampled {'label' if is_label else 'image'}: size {original_size}→{new_size}, spacing {original_spacing}→{new_spacing}")
    return sitk.GetArrayFromImage(resampled).transpose(2, 1, 0)

def resample_label_to_image(label_path, image_ref_path):
    """Resample label to match reference image (CT or input image)."""
    label_sitk = sitk.ReadImage(str(label_path))
    ref_sitk = sitk.ReadImage(str(image_ref_path))
    label_resampled = sitk.Resample(
        label_sitk,
        ref_sitk,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        label_sitk.GetPixelID()
    )
    print(f"Resampled label: {label_resampled.GetSize()} to match CT: {ref_sitk.GetSize()}")
    return sitk.GetArrayFromImage(label_resampled).transpose(2, 1, 0)

def window_ct_hu(ct_hu, level=50, width=350):
    lower, upper = level - width / 2.0, level + width / 2.0
    ct_clipped = np.clip(ct_hu, lower, upper)
    return (ct_clipped - lower) / (upper - lower + 1e-6)

def get_slice(volume, axis, idx):
    if axis == "axial":
        return volume[:, :, idx]
    elif axis == "coronal":
        return volume[:, idx, :]
    elif axis == "sagittal":
        return volume[idx, :, :]
    else:
        raise ValueError(f"Invalid axis: {axis}. Must be 'axial', 'coronal', or 'sagittal'.")

# ============================================================
# Overlay Builders
# ============================================================

def build_label_overlay(label_slice):
    unique_labels = np.unique(label_slice)
    if np.array_equal(unique_labels, [0]) or np.array_equal(unique_labels, [0, 1]):
        cmap = ListedColormap([[0,0,0,0], [1,0,0,0.45]])
        return label_slice.astype(np.int32), cmap, (0, 1)
    colors = [[0, 0, 0, 0]]
    rng = np.random.default_rng(42)
    for _ in range(int(unique_labels.max())):
        colors.append([*rng.random(3), 0.45])
    cmap = ListedColormap(colors)
    return label_slice.astype(np.int32), cmap, (0, int(unique_labels.max()))

def build_label_overlay_small_objects_3d(label_volume, min_size=200, connectivity=2):
    delete_small = RemoveSmallObjects(min_size=min_size, connectivity=connectivity)
    cleaned_label = delete_small(label_volume)
    small_objects = (label_volume > 0) & (cleaned_label == 0)
    large_objects = (cleaned_label > 0)
    overlay = np.zeros_like(label_volume, dtype=np.int32)
    overlay[large_objects] = 1
    overlay[small_objects] = 2
    cmap = ListedColormap([[0,0,0,0],[0,1,0,0.25],[1,0,0,0.25]])
    return overlay, cmap, (0, 2)

def build_label_overlay_largest_3d(label_volume):
    largest_component = KeepLargestConnectedComponent(connectivity=2)(label_volume)
    overlay = np.zeros_like(label_volume, dtype=np.int32)
    overlay[label_volume > 0] = 1
    overlay[largest_component > 0] = 2
    cmap = ListedColormap([[0,0,0,0],[1,0,0,0.25],[0,1,0,0.25]])
    return overlay, cmap, (0, 2)

# ============================================================
# Visualization
# ============================================================

def visualize_case(ct_path, label_path, uid, target,
                   axis="axial", mode="labels",
                   window_level=50, window_width=350,
                   save_dir=None, save_all_slices=False,
                   resample_labels=False,
                   resample_spacing=None):
    """
    Visualize one case with selectable mode:
      - 'labels', 'large_component', 'small_objects'
    Optionally resample labels to match CT grid.
    """
    if resample_spacing:
        ct = resample_image_to_spacing(ct_path, target_spacing=resample_spacing, is_label=False)
        label = resample_image_to_spacing(label_path, target_spacing=resample_spacing, is_label=True)
    else: 
        ct = load_volume(ct_path)
        if resample_labels:
            label = resample_label_to_image(label_path, ct_path)
        else:
            label = load_volume(label_path)
        #ct, label, _ = crop_to_label_region(ct[None,...], label, spacing= [0.73828125,0.73828125, 0.8999999761581421], margin_min=20 )
        #ct = ct[0]
        
    print(f"Loaded UID {uid}: CT shape {ct.shape}, Label shape {label.shape}")
    assert ct.shape == label.shape, f"Shape mismatch for {uid}"

    annotated_slices = np.where(np.any(label > 0, axis=(0, 1)))[0]
    if len(annotated_slices) > 0:
        print(f"Annotations at slices: {annotated_slices}")
    else:
        print("No annotations found.")

    if mode == "large_component":
        overlay_3d, cmap, vminmax = build_label_overlay_largest_3d(label)
    elif mode == "small_objects":
        overlay_3d, cmap, vminmax = build_label_overlay_small_objects_3d(label)
    else:
        overlay_3d, cmap, vminmax = label, *build_label_overlay(label[:, :, 0])[1:]

    axis_to_dim = {"axial": 2, "coronal": 1, "sagittal": 0}
    n_slices = ct.shape[axis_to_dim[axis]]

    out_uid_dir = None
    if save_all_slices and save_dir:
        out_uid_dir = Path(save_dir) / str(uid)
        out_uid_dir.mkdir(parents=True, exist_ok=True)

    def plot_slice(idx):
        ct_slice = get_slice(ct, axis, idx)
        overlay_slice = get_slice(overlay_3d, axis, idx)
        ct_img = window_ct_hu(ct_slice, window_level, window_width)

        plt.figure(figsize=(6,6))
        plt.imshow(ct_img.T, cmap="gray", origin="lower")
        plt.imshow(overlay_slice.T, cmap=cmap, origin="lower",
                   vmin=vminmax[0], vmax=vminmax[1])
        plt.axis("off")
        plt.title(f"UID {uid} | {target} | {axis.capitalize()} slice {idx}/{n_slices}")
        plt.tight_layout()

        if out_uid_dir:
            plt.savefig(out_uid_dir / f"{axis}_slice{idx:03d}.png", bbox_inches='tight')
            plt.close()
        else:
            plt.show()

    slice_slider = widgets.IntSlider(
        value=n_slices//2, min=0, max=n_slices-1, step=1,
        description=f'UID {uid}', continuous_update=False
    )
    widgets.interact(plot_slice, idx=slice_slider)

    if save_all_slices and out_uid_dir:
        print(f"Saving all slices for UID {uid} → {out_uid_dir}")
        for i in range(n_slices):
            plot_slice(i)

            plot_slice(i)
# ============================
# Main Visualization Function
# ============================


def visualize_dataset(uids, img_root, label_root,resample_labels=False,mode="label", axis="axial", save_dir=None, save_all_slices=False, resample_spacing=None):
    """Visualize multiple UIDs."""
    img_root, label_root = Path(img_root), Path(label_root)
    df = pd.read_csv(f"/data/colon_cancer/Classifier/ColonCancer/splits.csv")
    for uid in uids:
        # try naming patterns
        possible_img_names = [
            f"{uid}.nii.gz",f"{uid:03d}.nii.gz", f"{uid}_0000.nii.gz", f"{uid:03d}_0000.nii.gz",
            f"colon_{uid:03d}.nii.gz", f"{uid}.npy", f"{uid}.npz",  f"{uid:03d}.b2nd",
        ]
        possible_label_names = [
            f"{uid}.nii.gz",f"{uid:03d}.nii.gz", f"{uid}_0000.nii.gz", f"{uid:03d}_0000.nii.gz",
            f"colon_{uid:03d}.nii.gz", f"{uid}.npy", f"{uid}.npz",  f"{uid:03d}_seg.b2nd", 
        ]
        
        
        img_path = next((img_root / n for n in possible_img_names if (img_root / n).exists()), None)
        label_path = next((label_root / n for n in possible_label_names if (label_root / n).exists()), None)
        print(img_path, label_path)
        if img_path is None or label_path is None:
            print(f"Skipping UID {uid}: missing image or label")
            continue

        row = df[df["UID"] == uid]
        target = row["target"].values[0]
        if target == 0:
            target = "div"
        else:
            target = "cc"

        visualize_case(img_path, label_path, uid,mode=mode, axis=axis, target=target,
                       save_dir=save_dir, save_all_slices=save_all_slices,resample_labels=resample_labels, resample_spacing=resample_spacing)


# ============================
# Example usage
# ============================

if __name__ == "__main__":
    #108, 565, 752, 803, 370
    uids = [803 ]
    
    """
    #img_root = "/data/colon_cancer/Classifier/pp_Tr_npz"
    #label_root = "/data/colon_cancer/Classifier/resampledTr/labels_resampled"
    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/nnUNet_results/Dataset101_CC/nnUNetTrainer__nnUNetResEncUNetLPlans__3d_fullres/fold_0/validation"
    save_dir = "/data/benchaaben/classifier/ct_overlays/finetuned"

    visualize_dataset(uids, img_root, label_root,
                    axis="axial", save_dir=save_dir, mode ="label", save_all_slices=False)
    
    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/final"
    save_dir = "/data/benchaaben/classifier/ct_overlays/pretrained"
    visualize_dataset(uids, img_root, label_root,
                      axis="axial", save_dir=save_dir,mode ="label",resample_labels=True, save_all_slices=False)


    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/original"
    save_dir = "/data/benchaaben/classifier/ct_overlays/pretrained"
    visualize_dataset(uids, img_root, label_root,mode ="label",
                      axis="axial", save_dir=save_dir, save_all_slices=False)
    
    img_root = "/data/colon_cancer/nnUNet_raw/Dataset100_CC/imagesTr"
    label_root = "/data/colon_cancer/nnUNet_raw/Dataset100_CC/labelsTr"
    save_dir = "/data/benchaaben/classifier/ct_overlays/original"
    visualize_dataset(uids, img_root, label_root,mode ="label",
                      axis="axial", save_dir=save_dir, save_all_slices=False)
    """
    ##input to segmentator
    #img_root = f"/data/colon_cancer/Classifier/ColonCancer/nnUNetPlans_3d_fullres"
    #label_root = f"/data/colon_cancer/Classifier/ColonCancer/nnUNetPlans_3d_fullres"
    #visualize_dataset(uids, img_root, label_root,mode ="label",
    #                  axis="axial", save_all_slices=False,resample_spacing=None)
    
    
    ##input to classifier 
    #img_root = f"/data/colon_cancer/Classifier/ColonCancer/resampledTr/images_resampled"
    #label_root = f"/data/colon_cancer/Classifier/ColonCancer/resampledTr/labels_resampled"
    #visualize_dataset(uids, img_root, label_root,mode ="label",
    #                  axis="axial", save_all_slices=False)

    img_root = f"/data/colon_cancer/nnUNet_raw/Dataset100_CC/imagesTr"
    label_root=f"/data/colon_cancer/nnUNet_preprocessed_old/Dataset100_CC/gt_segmentations"
    visualize_dataset(uids, img_root, label_root,mode ="label",
                      axis="axial", save_all_slices=False,resample_spacing=None)

    


/data/colon_cancer/nnUNet_raw/Dataset100_CC/imagesTr/803_0000.nii.gz /data/colon_cancer/nnUNet_preprocessed_old/Dataset100_CC/gt_segmentations/803.nii.gz
Loaded UID 803: CT shape (512, 512, 701), Label shape (512, 512, 701)
Annotations at slices: [120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137
 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155
 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170]


interactive(children=(IntSlider(value=350, continuous_update=False, description='UID 803', max=700), Output())…